In [32]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from xgboost import XGBClassifier, XGBRegressor

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression, LinearRegression, SGDClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.naive_bayes import GaussianNB

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report,
    mean_absolute_error, mean_squared_error, r2_score
)

from IPython.display import FileLink

In [33]:
df = pd.read_csv("output.csv")

# Convert numeric columns
numeric_to_clean = [
    "BOARDINGS", "ALIGHTINGS", "TRIPS",
    "AVG_BOARDINGS", "AVG_ALIGHTINGS", "AVG_ACTIVITY",
    "PASS_LOAD", "PEAK_LOAD", "AVG_PEAK_LOAD"
]

for col in numeric_to_clean:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Create TRIP_HOUR if TRIP_TIME exists
if "TRIP_TIME" in df.columns:
    df["TRIP_TIME"] = df["TRIP_TIME"].astype(str)
    df["TRIP_HOUR"] = pd.to_datetime(df["TRIP_TIME"], errors="coerce").dt.hour

print(df.shape)
df.head()

C:\Users\balto\AppData\Local\Temp\ipykernel_34680\4227888796.py:1: DtypeWarning: Columns (24) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("output.csv")
C:\Users\balto\AppData\Local\Temp\ipykernel_34680\4227888796.py:16: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["TRIP_HOUR"] = pd.to_datetime(df["TRIP_TIME"], errors="coerce").dt.hour


(388202, 33)


,ROUTE_NAME,ROUTE_NUMBER,SERVICE_PERIOD,SERVICE_CODE,DIRECTION_NAME,BRANCH,TRIP_TIME,SORT_ORDER,STOP_ID,MAIN_CROSS_STREET,...,STOP_DISPLAY,Additional_Notes,PATTERN_KEY,BLOCK,TOTAL_SORT,SORT_SP,ROUTE_REV,SERVICE_CODE2,Stop_ID_REV,TRIP_HOUR
0,22: Palo Alto - Eastridge,22.0,Saturday,Frequent,EAST,[22]Palo Alto > Eastridge,00:35:00,680.0,1.0,SANTA CLARA TRANSIT CENTER,...,1.0,NaN,EB04,3507.0,STATS,2.0,22:,Local,60001:,0.0
1,22: Palo Alto - Eastridge,22.0,Saturday,Frequent,EAST,[22]Palo Alto > Eastridge,01:57:00,680.0,1.0,SANTA CLARA TRANSIT CENTER,...,1.0,NaN,EB04,2707.0,STATS,2.0,22:,Local,60001:,1.0
2,22: Palo Alto - Eastridge,22.0,Saturday,Frequent,EAST,[22]Palo Alto > Eastridge,04:32:00,680.0,1.0,SANTA CLARA TRANSIT CENTER,...,1.0,NaN,EB04,2207.0,STATS,2.0,22:,Local,60001:,4.0
3,22: Palo Alto - Eastridge,22.0,Saturday,Frequent,EAST,[22]Palo Alto > Eastridge,04:58:00,680.0,1.0,SANTA CLARA TRANSIT CENTER,...,1.0,NaN,EB04,2407.0,STATS,2.0,22:,Local,60001:,4.0
4,22: Palo Alto - Eastridge,22.0,Saturday,Frequent,EAST,[22]Palo Alto > Eastridge,05:23:00,680.0,1.0,SANTA CLARA TRANSIT CENTER,...,1.0,NaN,EB04,2607.0,STATS,2.0,22:,Local,60001:,5.0


In [34]:
df["ridership_class"] = pd.cut(
    df["BOARDINGS"],
    bins=[-1, 5, 25, float("inf")],
    labels=["Low", "Medium", "High"]
)

df = df.dropna(subset=["ridership_class", "BOARDINGS"])

print(df["ridership_class"].value_counts())

ridership_class
Low       307205
Medium     61506
High       14003
Name: count, dtype: int64


In [35]:
features = [
    "ROUTE_NAME",
    "ROUTE_NUMBER",
    "SERVICE_PERIOD",
    "SERVICE_CODE",
    "DIRECTION_NAME",
    "BRANCH",
    "TRIPS",
    "AVG_BOARDINGS",
    "AVG_ALIGHTINGS",
    "AVG_ACTIVITY",
    "PASS_LOAD",
    "PEAK_LOAD",
    "AVG_PEAK_LOAD",
    "CITY"
]

target = "ridership_class"

X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape, X_test.shape)

(306171, 14) (76543, 14)


In [36]:
numeric_cols = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_cols),
    ("cat", categorical_pipeline, categorical_cols)
])

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Preprocessing complete")

Preprocessing complete


In [37]:
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)

models = {
    "Logistic Regression SGD": SGDClassifier(loss="log_loss", max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=12, random_state=42),
    "Random Forest": RandomForestClassifier(
        n_estimators=50,
        max_depth=12,
        n_jobs=-1,
        random_state=42
    ),
    "Gaussian Naive Bayes": GaussianNB()
}

results = {}

for name, model in models.items():

    if name == "Gaussian Naive Bayes":
        X_train_model = X_train_processed.toarray() if hasattr(X_train_processed, "toarray") else X_train_processed
        X_test_model = X_test_processed.toarray() if hasattr(X_test_processed, "toarray") else X_test_processed
    else:
        X_train_model = X_train_processed
        X_test_model = X_test_processed

    model.fit(X_train_model, y_train)
    y_pred = model.predict(X_test_model)

    results[name] = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision_weighted": precision_score(y_test, y_pred, average="weighted", zero_division=0),
        "Recall_weighted": recall_score(y_test, y_pred, average="weighted", zero_division=0),
        "F1_weighted": f1_score(y_test, y_pred, average="weighted", zero_division=0)
    }

    print("\n==============================")
    print(name)
    print("==============================")
    print(classification_report(y_test, y_pred, zero_division=0))


Logistic Regression SGD
              precision    recall  f1-score   support

        High       0.92      0.83      0.87      2801
         Low       0.94      1.00      0.97     61441
      Medium       0.92      0.68      0.78     12301

    accuracy                           0.94     76543
   macro avg       0.93      0.84      0.87     76543
weighted avg       0.94      0.94      0.93     76543


Decision Tree
              precision    recall  f1-score   support

        High       1.00      1.00      1.00      2801
         Low       1.00      1.00      1.00     61441
      Medium       1.00      1.00      1.00     12301

    accuracy                           1.00     76543
   macro avg       1.00      1.00      1.00     76543
weighted avg       1.00      1.00      1.00     76543


Random Forest
              precision    recall  f1-score   support

        High       0.98      0.34      0.50      2801
         Low       0.97      0.99      0.98     61441
      Medium       0

In [38]:
metrics_df = pd.DataFrame(results).T.reset_index()
metrics_df = metrics_df.rename(columns={"index": "Model"})

metrics_df = metrics_df[
    ["Model", "Accuracy", "Precision_weighted", "Recall_weighted", "F1_weighted"]
]

metrics_df.to_csv("ml_model_metrics.csv", index=False)

display(metrics_df)
FileLink("ml_model_metrics.csv")

,Model,Accuracy,Precision_weighted,Recall_weighted,F1_weighted
0,Logistic Regression SGD,0.938845,0.937623,0.938845,0.934878
1,Decision Tree,0.999961,0.999961,0.999961,0.999961
2,Random Forest,0.942999,0.943969,0.942999,0.937581
3,Gaussian Naive Bayes,0.110513,0.776351,0.110513,0.138448


e:\Masters Data Analytics\VTA-s-2025-Ridership-by-Stop-Station-\ml_model_metrics.csv

In [39]:
best_clf_name = metrics_df.sort_values("F1_weighted", ascending=False).iloc[0]["Model"]
best_clf = models[best_clf_name]

X_all_processed = preprocessor.transform(X)

if best_clf_name == "Gaussian Naive Bayes":
    X_all_model = X_all_processed.toarray() if hasattr(X_all_processed, "toarray") else X_all_processed
else:
    X_all_model = X_all_processed

df["final_predicted_ridership_class"] = best_clf.predict(X_all_model)
df["ml_target_ridership_class"] = df["ridership_class"]

print("Best classification model:", best_clf_name)
df.head()

Best classification model: Decision Tree


,ROUTE_NAME,ROUTE_NUMBER,SERVICE_PERIOD,SERVICE_CODE,DIRECTION_NAME,BRANCH,TRIP_TIME,SORT_ORDER,STOP_ID,MAIN_CROSS_STREET,...,BLOCK,TOTAL_SORT,SORT_SP,ROUTE_REV,SERVICE_CODE2,Stop_ID_REV,TRIP_HOUR,ridership_class,final_predicted_ridership_class,ml_target_ridership_class
0,22: Palo Alto - Eastridge,22.0,Saturday,Frequent,EAST,[22]Palo Alto > Eastridge,00:35:00,680.0,1.0,SANTA CLARA TRANSIT CENTER,...,3507.0,STATS,2.0,22:,Local,60001:,0.0,Low,Low,Low
1,22: Palo Alto - Eastridge,22.0,Saturday,Frequent,EAST,[22]Palo Alto > Eastridge,01:57:00,680.0,1.0,SANTA CLARA TRANSIT CENTER,...,2707.0,STATS,2.0,22:,Local,60001:,1.0,Low,Low,Low
2,22: Palo Alto - Eastridge,22.0,Saturday,Frequent,EAST,[22]Palo Alto > Eastridge,04:32:00,680.0,1.0,SANTA CLARA TRANSIT CENTER,...,2207.0,STATS,2.0,22:,Local,60001:,4.0,Low,Low,Low
3,22: Palo Alto - Eastridge,22.0,Saturday,Frequent,EAST,[22]Palo Alto > Eastridge,04:58:00,680.0,1.0,SANTA CLARA TRANSIT CENTER,...,2407.0,STATS,2.0,22:,Local,60001:,4.0,Low,Low,Low
4,22: Palo Alto - Eastridge,22.0,Saturday,Frequent,EAST,[22]Palo Alto > Eastridge,05:23:00,680.0,1.0,SANTA CLARA TRANSIT CENTER,...,2607.0,STATS,2.0,22:,Local,60001:,5.0,Low,Low,Low


In [47]:
# =========================
# Regression Models + predicted_boardings - FIXED
# =========================

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import numpy as np
import pandas as pd
from IPython.display import FileLink

# Make sure BOARDINGS is numeric
df["BOARDINGS"] = pd.to_numeric(df["BOARDINGS"], errors="coerce")

# Drop rows where regression target is missing
df_reg = df.dropna(subset=["BOARDINGS"]).copy()

# Use features that exist in the dataframe
reg_features = [
    "ROUTE_NAME",
    "ROUTE_NUMBER",
    "SERVICE_PERIOD",
    "SERVICE_CODE",
    "DIRECTION_NAME",
    "BRANCH",
    "TRIPS",
    "AVG_ALIGHTINGS",
    "AVG_ACTIVITY",
    "PASS_LOAD",
    "PEAK_LOAD",
    "AVG_PEAK_LOAD",
    "CITY"
]

reg_features = [col for col in reg_features if col in df_reg.columns]

X_reg = df_reg[reg_features]
y_reg = df_reg["BOARDINGS"]

Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    X_reg,
    y_reg,
    test_size=0.2,
    random_state=42
)

# Separate regression preprocessor
reg_numeric_cols = Xr_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
reg_categorical_cols = Xr_train.select_dtypes(include=["object", "category"]).columns.tolist()

reg_numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

reg_categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

reg_preprocessor = ColumnTransformer([
    ("num", reg_numeric_pipeline, reg_numeric_cols),
    ("cat", reg_categorical_pipeline, reg_categorical_cols)
])

# Fit regression preprocessor once
Xr_train_p = reg_preprocessor.fit_transform(Xr_train)
Xr_test_p = reg_preprocessor.transform(Xr_test)

reg_models = {
    "Linear Regression": LinearRegression(),
    "Random Forest Regressor": RandomForestRegressor(
        n_estimators=50,
        max_depth=12,
        n_jobs=-1,
        random_state=42
    ),
    "XGBoost Regressor": XGBRegressor(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        tree_method="hist",
        device="cuda",
        random_state=42
    )
}

reg_results = {}

for name, model in reg_models.items():
    try:
        model.fit(Xr_train_p, yr_train)
        y_pred_reg = model.predict(Xr_test_p)

        reg_results[name] = {
            "MAE": mean_absolute_error(yr_test, y_pred_reg),
            "RMSE": np.sqrt(mean_squared_error(yr_test, y_pred_reg)),
            "R2": r2_score(yr_test, y_pred_reg)
        }

    except Exception as e:
        print(f"{name} failed: {e}")

reg_metrics_df = pd.DataFrame(reg_results).T.reset_index()
reg_metrics_df = reg_metrics_df.rename(columns={"index": "Model"})

display(reg_metrics_df)

# =========================
# Create predicted_boardings column
# =========================

best_reg_name = reg_metrics_df.sort_values("R2", ascending=False).iloc[0]["Model"]
best_reg = reg_models[best_reg_name]

X_reg_all_p = reg_preprocessor.transform(X_reg)

df_reg["predicted_boardings"] = best_reg.predict(X_reg_all_p)

# Optional: prevent negative predicted boardings
df_reg["predicted_boardings"] = df_reg["predicted_boardings"].clip(lower=0)

print("Best regression model:", best_reg_name)
display(df_reg[["ROUTE_NAME", "BOARDINGS", "predicted_boardings"]].head())

# =========================
# Export files for Looker Studio
# =========================

reg_metrics_df.to_csv("ml_regression_metrics.csv", index=False)
df_reg.to_csv("output_cleaned_ml_predictions_with_regression.csv", index=False)

print("Saved: ml_regression_metrics.csv")
print("Saved: output_cleaned_ml_predictions_with_regression.csv")

display(FileLink("ml_regression_metrics.csv"))
display(FileLink("output_cleaned_ml_predictions_with_regression.csv"))

,Model,MAE,RMSE,R2
0,Linear Regression,3.412074,6.167019,0.705835
1,Random Forest Regressor,0.244660,1.354828,0.985803
2,XGBoost Regressor,0.349843,1.666736,0.978513


Best regression model: Random Forest Regressor


,ROUTE_NAME,BOARDINGS,predicted_boardings
0,25: De Anza Col - Alum Rock Trans Ctr,5.0,5.204281
1,22: Palo Alto - Eastridge,12.0,11.385284
2,60: Milpitas BART - Winchester Station,3.0,3.000000
3,61: Good Sam Hosp - Sierra & Piedmont,0.0,0.000000
4,26: West Valley College - Eastridge,1.0,0.940636


Saved: ml_regression_metrics.csv
Saved: output_cleaned_ml_predictions_with_regression.csv


e:\Masters Data Analytics\VTA-s-2025-Ridership-by-Stop-Station-\ml_regression_metrics.csv

e:\Masters Data Analytics\VTA-s-2025-Ridership-by-Stop-Station-\output_cleaned_ml_predictions_with_regression.csv

In [41]:
reg_metrics_df.to_csv("ml_regression_metrics.csv", index=False)

FileLink("ml_regression_metrics.csv")

e:\Masters Data Analytics\VTA-s-2025-Ridership-by-Stop-Station-\ml_regression_metrics.csv

In [42]:
# =========================
# Add Predicted Boardings - FIXED
# =========================

best_reg_name = reg_metrics_df.sort_values("R2", ascending=False).iloc[0]["Model"]
best_reg = reg_models[best_reg_name]

# IMPORTANT: use reg_preprocessor, not preprocessor
X_reg_all_p = reg_preprocessor.transform(X_reg)

df["predicted_boardings"] = best_reg.predict(X_reg_all_p)

print("Best regression model:", best_reg_name)
df[["ROUTE_NAME", "BOARDINGS", "predicted_boardings"]].head()

Best regression model: Random Forest Regressor


,ROUTE_NAME,BOARDINGS,predicted_boardings
0,22: Palo Alto - Eastridge,2.0,2.0
1,22: Palo Alto - Eastridge,0.0,0.0
2,22: Palo Alto - Eastridge,0.0,0.0
3,22: Palo Alto - Eastridge,0.0,0.0
4,22: Palo Alto - Eastridge,1.0,1.0


In [43]:
df.to_csv("output_cleaned_ml_predictions_with_regression.csv", index=False)

FileLink("output_cleaned_ml_predictions_with_regression.csv")

e:\Masters Data Analytics\VTA-s-2025-Ridership-by-Stop-Station-\output_cleaned_ml_predictions_with_regression.csv

In [44]:
# =========================
# Fix Time Column for Looker Studio
# =========================

import pandas as pd

# Load your dataset (NOT metrics file)
df = pd.read_csv("output_cleaned_ml_predictions_SAMPLE.csv")

# Step 1: Ensure TRIP_HOUR is numeric
df["TRIP_HOUR"] = pd.to_numeric(df["TRIP_HOUR"], errors="coerce")

# Step 2: Create proper datetime column
df["trip_datetime"] = pd.to_datetime("2024-01-01") + pd.to_timedelta(df["TRIP_HOUR"], unit="h")

# Step 3: (Optional but recommended) Drop bad rows
df = df.dropna(subset=["trip_datetime"])

# Step 4: Export new file
output_file = "output_with_time_fixed.csv"
df.to_csv(output_file, index=False)

print(f"Saved: {output_file}")
df.head()

Saved: output_with_time_fixed.csv


,ROUTE_NAME,ROUTE_NUMBER,SERVICE_PERIOD,SERVICE_CODE,DIRECTION_NAME,BRANCH,TRIP_TIME,STOP_ID,MAIN_CROSS_STREET,BOARDINGS,...,CITY,TRIP_HOUR,ridership_class,pred_logistic_regression_sgd,pred_decision_tree,pred_random_forest,pred_gaussian_naive_bayes,final_predicted_ridership_class,ml_target_ridership_class,trip_datetime
0,25: De Anza Col - Alum Rock Trans Ctr,25.0,Weekday,Frequent,EAST,[25]De Anza > Alum Rock,10:02:00,769.0,STORY + MCGINNESS,5.0,...,San Jose,10.033333,Low,Low,Low,Low,Low,Low,Low,2024-01-01 10:01:59.999999998
1,22: Palo Alto - Eastridge,22.0,Weekday,Frequent,WEST,[22]Eastridge > Palo Alto,09:43:00,490.0,EL CAMINO + SYLVAN,12.0,...,Mountain View,9.716667,Medium,Medium,Medium,Medium,Low,Medium,Medium,2024-01-01 09:43:00.000000001
2,60: Milpitas BART - Winchester Station,60.0,Weekday,Frequent,NORTH,[60]Winchester TC > Milpitas BART,13:05:00,2707.0,WINCHESTER + RINCON,3.0,...,Campbell,13.083333,Low,Low,Low,Low,Low,Low,Low,2024-01-01 13:04:59.999999998
3,61: Good Sam Hosp - Sierra & Piedmont,61.0,Weekday,Frequent,SOUTH,[61]Sier/Pied > GoodSam via Bascom,08:31:00,2890.0,BASCOM + SHELLEY,0.0,...,San Jose,8.516667,Low,Low,Low,Low,Low,Low,Low,2024-01-01 08:31:00.000000001
4,26: West Valley College - Eastridge,26.0,Weekday,Frequent,WEST,[26]Eastridge > West Valley,14:18:00,2619.0,SARATOGA + COX,1.0,...,Saratoga,14.300000,Low,Low,Low,Low,Low,Low,Low,2024-01-01 14:18:00.000000000
